In [20]:
import os
import tempfile
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from tqdm.auto import tqdm

from datetime import datetime

from xgboost import XGBClassifier

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
)

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer


from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import VarianceThreshold

from sklearn.utils.class_weight import compute_sample_weight
import inspect


import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
df = pd.read_csv("../data/processed/validated_data.csv")

import numpy as np
from datetime import datetime

current_year = datetime.now().year

# 1. core time-based feature

df["vehicleAge"] = current_year - df["yearOfRegistration"]
df = df.drop(columns=["yearOfRegistration"])


# 2. usage intensity features

df["kmPerYear"] = df["kilometer"] / (df["vehicleAge"] + 1)
df["km_power_ratio"] = df["kilometer"] / (df["power"] + 1)


# 3. power transformations (reduce skew)

df["log_power"] = np.log1p(df["power"])
df["power_per_age"] = df["power"] / (df["vehicleAge"] + 1)


# 4. mileage transformation

df["log_kilometer"] = np.log1p(df["kilometer"])


# 5. behavioral flags

df["is_vintage"] = (df["vehicleAge"] > 25).astype(int)
df["high_mileage_flag"] = (df["kmPerYear"] > df["kmPerYear"].quantile(0.75)).astype(int)


# 6. drop redundant raw columns
df = df.drop(columns=["yearOfRegistration"], errors="ignore")

print(f"After feature engineering: {df.shape}")

print("\nEngineered features added:")

new_features = [
    "vehicleAge",
    "kmPerYear",
    "km_power_ratio",
    "log_power",
    "power_per_age",
    "log_kilometer",
    "is_vintage",
    "high_mileage_flag",
]

for f in new_features:
    print(f"  - {f}")
    

After feature engineering: (245630, 17)

Engineered features added:
  - vehicleAge
  - kmPerYear
  - km_power_ratio
  - log_power
  - power_per_age
  - log_kilometer
  - is_vintage
  - high_mileage_flag


In [22]:
# drop leakage source
df = df.drop(columns=["price"])

# rare category grouping
model_counts = df["model"].value_counts(normalize=True)
rare_models = model_counts[model_counts < 0.001].index
df["model"] = df["model"].apply(lambda x: "other" if x in rare_models else x)

# encode target
df["price_tier"] = df["price_tier"].map({"budget": 0, "mid-range": 1, "luxury": 2})

X = df.drop(columns=["price_tier"])
y = df["price_tier"]

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('Class distribution (train):')
print(y_train.value_counts().sort_index())
print(y_train.value_counts(normalize=True).sort_index().round(4))

sample_weight_train = compute_sample_weight(class_weight='balanced', y=y_train)

Class distribution (train):
price_tier
0    103958
1     75556
2     16990
Name: count, dtype: int64
price_tier
0    0.5290
1    0.3845
2    0.0865
Name: proportion, dtype: float64


In [24]:
feature_selector = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1),
    threshold="median",
)

In [25]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in self.cols:
            self.freq_maps[col] = X[col].value_counts(normalize=True)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            X[col + "_freq"] = X[col].map(self.freq_maps[col]).fillna(0)
        return X

    def get_feature_names_out(self, input_features=None):
        return [col + "_freq" for col in self.cols]

In [26]:
enc = OneHotEncoder(handle_unknown="ignore")
enc.fit(X_train[["seller"]])

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_categ

In [27]:
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = ['vehicleType', 'fuelType', 'seller']

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),

    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
    ]), cat_cols)
])

In [ ]:
def business_metrics(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    luxury_recall = report.get("2", {}).get("recall", 0.0)

    y_t = pd.Series(y_true).reset_index(drop=True)
    y_p = pd.Series(y_pred).reset_index(drop=True)

    severe = ((y_t == 0) & (y_p == 2)) | ((y_t == 2) & (y_p == 0))
    severe_rate = severe.mean()

    return luxury_recall, severe_rate

In [ ]:
model_configs = {
    "baseline_dummy": {
        "estimator": DummyClassifier(),
        "param_distributions": {
            "model__strategy": ["most_frequent", "prior", "stratified"]
        },
        "n_iter": 3,
    },
    "log_reg": {
        "estimator": LogisticRegression(max_iter=2500, solver="saga", random_state=42),
        "param_distributions": {"model__C": np.logspace(-2, 1, 6)},
        "n_iter": 6,
    },
    "linear_svc": {
        "estimator": LinearSVC(random_state=42),
        "param_distributions": {
            "model__C": np.logspace(-3, 2, 6),
            "model__loss": ["hinge", "squared_hinge"],
        },
        "n_iter": 8,
    },
    "random_forest": {
        "estimator": RandomForestClassifier(random_state=42, n_jobs=1),
        "param_distributions": {
            "model__n_estimators": [100, 200, 300, 500],
            "model__max_depth": [None, 10, 20, 30],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2"],
            "model__bootstrap": [True, False],
        },
        "n_iter": 8,
    },
    "extra_trees": {
        "estimator": ExtraTreesClassifier(random_state=42, n_jobs=1),
        "param_distributions": {
            "model__n_estimators": [100, 200, 300, 500],
            "model__max_depth": [None, 10, 20, 30],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2"],
            "model__bootstrap": [False, True],
        },
        "n_iter": 8,
    },
    "decision_tree": {
        "estimator": DecisionTreeClassifier(random_state=42),
        "param_distributions": {
            "model__max_depth": [None, 5, 10, 20, 30],
            "model__min_samples_split": [2, 5, 10, 20],
            "model__min_samples_leaf": [1, 2, 4, 8],
            "model__criterion": ["gini", "entropy"],
            "model__max_features": [None, "sqrt", "log2"],
        },
        "n_iter": 8,
    },
    "xgboost": {
        "estimator": XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            random_state=42,
            n_jobs=1,
            tree_method="hist",
        ),
        "param_distributions": {
            "model__n_estimators": [200, 300, 500],
            "model__max_depth": [4, 6, 8, 10],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__subsample": [0.7, 0.8, 1.0],
            "model__colsample_bytree": [0.7, 0.8, 1.0],
            "model__gamma": [0, 1, 5],
        },
        "n_iter": 8,
    },
}

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("used_car_price_tier_classification")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

best_estimators = {}
best_scores = {}

mlflow.end_run()

In [ ]:
def build_pipeline(estimator, strategy, use_feature_selection=False):
    steps = [
        ("freq", FrequencyEncoder(["brand", "model"])),
        (
            "drop_cols",
            FunctionTransformer(
                lambda X: X.drop(columns=["brand", "model"]), validate=False
            ),
        ),
        ("preprocess", preprocess),
    ]

    if use_feature_selection:
        steps.append(("feature_selection", feature_selector))
    if strategy == "smote":
        steps.append(("resample", SMOTE(random_state=42)))
    elif strategy == "undersample":
        steps.append(("resample", RandomUnderSampler(random_state=42)))

    steps.append(("model", estimator))

    return Pipeline(steps)

In [ ]:
def run_experiments(model_configs, X_train, y_train, X_test, y_test, cv, strategy):
    results = []
    best_estimators = {}

    mlflow.end_run()

    for model_name, cfg in tqdm(model_configs.items(), desc=f"Training ({strategy})"):

        pipe = build_pipeline(cfg["estimator"], strategy)

        search = RandomizedSearchCV(
            pipe,
            cfg["param_distributions"],
            n_iter=cfg["n_iter"],
            scoring={"macro_f1": "f1_macro", "balanced_accuracy": "balanced_accuracy"},
            refit="macro_f1",
            cv=cv,
            n_jobs=1,
            random_state=42,
        )

        with mlflow.start_run(run_name=f"{model_name}_{strategy}"):

            search.fit(X_train, y_train)

            best_model = search.best_estimator_
            best_idx = search.best_index_

            cv_f1 = search.cv_results_["mean_test_macro_f1"][best_idx]
            cv_bal = search.cv_results_["mean_test_balanced_accuracy"][best_idx]

            y_pred = best_model.predict(X_test)

            luxury_recall, severe_rate = business_metrics(y_test, y_pred)

            cm = confusion_matrix(y_test, y_pred)

            plt.figure()
            sns.heatmap(cm, annot=True, fmt="d")
            plt.xlabel("Predicted")
            plt.ylabel("Actual")
            plt.title(f"{model_name} - {strategy}")

            with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
                plt.savefig(tmp.name)
                mlflow.log_artifact(tmp.name, "confusion_matrix")

            plt.close()

            # logging params
            mlflow.log_param("model", model_name)
            mlflow.log_param("strategy", strategy)

            for k, v in search.best_params_.items():
                mlflow.log_param(k, v)

            mlflow.log_param("cv_folds", cv.get_n_splits())

            # metrics
            mlflow.log_metric("cv_f1", cv_f1)
            mlflow.log_metric("cv_bal_acc", cv_bal)
            mlflow.log_metric("test_luxury_recall", luxury_recall)
            mlflow.log_metric("test_severe_rate", severe_rate)

            mlflow.sklearn.log_model(best_model, "model")

            results.append(
                {
                    "model": model_name,
                    "strategy": strategy,
                    "cv_f1": cv_f1,
                    "cv_bal_acc": cv_bal,
                    "test_luxury_recall": luxury_recall,
                    "test_severe_rate": severe_rate,
                }
            )

            best_estimators[f"{model_name}_{strategy}"] = best_model

    return pd.DataFrame(results), best_estimators

In [ ]:
results_smote, _ = run_experiments(
    model_configs, X_train, y_train, X_test, y_test, cv, strategy="smote"
)

results_under, _ = run_experiments(
    model_configs, X_train, y_train, X_test, y_test, cv, strategy="undersample"
)

results_none, _ = run_experiments(
    model_configs, X_train, y_train, X_test, y_test, cv, strategy="none"
)

In [ ]:
all_results = pd.concat([results_none, results_smote, results_under])
print(all_results.sort_values("test_luxury_recall", ascending=False))

In [ ]:
# results_df = pd.DataFrame(results)
# results_df

In [ ]:
# results_df = pd.DataFrame(results).sort_values(
#     ["test_luxury_recall", "test_severe_misclassification_rate"],
#     ascending=[False, True],
# )
# print(results_df)

# best_model_name = results_df.iloc[0]['model']
# best_model = best_estimators[best_model_name]

# best_model.fit(X_train, y_train)

## Test Evaluation

In [ ]:
# y_test_pred = best_model.predict(X_test)

# test_macro_f1 = f1_score(y_test, y_test_pred, average="macro")
# test_bal_acc = balanced_accuracy_score(y_test, y_test_pred)
# test_luxury_recall, test_severe_rate = business_metrics(y_test, y_test_pred)

# print("\nFinal Test Results:")
# print("Model:", best_model_name)
# print("Macro F1:", test_macro_f1)
# print("Balanced Accuracy:", test_bal_acc)
# print("Luxury Recall:", test_luxury_recall)
# print("Severe Error Rate:", test_severe_rate)

# print("\nClassification Report:")
# print(classification_report(y_test, y_test_pred, zero_division=0))

## Log Final Model

In [ ]:
# with mlflow.start_run(run_name=f"final_{best_model_name}"):

#     mlflow.log_param("selected_model", best_model_name)
#     mlflow.log_metric("test_macro_f1", test_macro_f1)
#     mlflow.log_metric("test_balanced_accuracy", test_bal_acc)
#     mlflow.log_metric("test_luxury_recall", test_luxury_recall)
#     mlflow.log_metric("test_severe_rate", test_severe_rate)

#     mlflow.sklearn.log_model(best_model, "model")